In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv("/Users/sreeharivenkataraman/Personal Files/DS_Projects/project_2/dataset/online_food_delivery_preprocessed.csv")

In [3]:
df

,Order_ID,Customer_ID,Customer_Age,Customer_Gender,City,Area,Restaurant_ID,Restaurant_Name,Cuisine_Type,Order_Date,...,Payment_Mode,Order_Status,Cancellation_Reason,Delivery_Partner_ID,Delivery_Rating,Restaurant_Rating,Order_Day,Peak_Hour,Profit_Margin,Profit_Margin_Percentage
0,ORD000001,CUST6948,19.0,Male,Unknown,Central,RES936,Restaurant_29,Chinese,2024-10-20,...,UPI,Delivered,Not Applicable,DP563,5.0,4.4,Weekend,Peak,0.13,13.0
1,ORD000002,CUST6515,39.0,Female,Chennai,North,RES689,Restaurant_419,Chinese,2024-08-12,...,COD,Delivered,Not Applicable,DP369,5.0,4.7,Weekday,Peak,0.48,48.0
2,ORD000003,CUST1765,39.0,Male,Delhi,Unknown,RES723,Restaurant_244,Arabian,2024-12-08,...,Wallet,Delivered,Not Applicable,DP580,4.0,4.9,Weekend,Peak,0.08,8.0
3,ORD000004,CUST2744,39.0,Male,Mumbai,Central,RES951,Restaurant_178,Chinese,2024-10-08,...,UPI,Cancelled,Late Delivery,DP155,NaN,3.4,Weekday,Unknown,0.04,4.0
4,ORD000005,CUST4389,57.0,Female,Chennai,South,RES419,Restaurant_262,Chinese,2024-02-04,...,Card,Delivered,Not Applicable,DP728,2.0,4.4,Weekend,Non-Peak,0.12,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,ORD099996,CUST3722,39.0,Unknown,Mumbai,Unknown,RES295,Restaurant_500,Arabian,2024-06-10,...,Card,Delivered,Not Applicable,DP835,2.0,4.0,Weekday,Peak,0.17,17.0
99996,ORD099997,CUST6158,39.0,Male,Mumbai,East,RES157,Restaurant_458,Arabian,2024-04-22,...,Unknown,Delivered,Not Applicable,DP476,4.0,3.5,Weekday,Non-Peak,0.25,25.0
99997,ORD099998,CUST2155,39.0,Unknown,Delhi,East,RES287,Restaurant_273,Unknown,2024-02-23,...,COD,Cancelled,Unknown,DP577,NaN,4.0,Weekday,Peak,0.47,47.0
99998,ORD099999,CUST3015,39.0,Female,Delhi,North,RES491,Restaurant_241,Unknown,2024-12-10,...,Unknown,Delivered,Not Applicable,DP896,3.0,4.3,Weekday,Unknown,0.44,44.0


In [4]:
def check_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print("Column:", column)
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Outlier Count:", len(outliers))

    return outliers

In [5]:
check_outliers(df, "Order_Value")

Column: Order_Value
Q1: 935.0
Q3: 2743.0
IQR: 1808.0
Lower Bound: -1777.0
Upper Bound: 5455.0
Outlier Count: 0


,Order_ID,Customer_ID,Customer_Age,Customer_Gender,City,Area,Restaurant_ID,Restaurant_Name,Cuisine_Type,Order_Date,...,Payment_Mode,Order_Status,Cancellation_Reason,Delivery_Partner_ID,Delivery_Rating,Restaurant_Rating,Order_Day,Peak_Hour,Profit_Margin,Profit_Margin_Percentage


In [6]:
check_outliers(df, "Delivery_Time_Min")

Column: Delivery_Time_Min
Q1: 58.0
Q3: 165.0
IQR: 107.0
Lower Bound: -102.5
Upper Bound: 325.5
Outlier Count: 0


,Order_ID,Customer_ID,Customer_Age,Customer_Gender,City,Area,Restaurant_ID,Restaurant_Name,Cuisine_Type,Order_Date,...,Payment_Mode,Order_Status,Cancellation_Reason,Delivery_Partner_ID,Delivery_Rating,Restaurant_Rating,Order_Day,Peak_Hour,Profit_Margin,Profit_Margin_Percentage


In [8]:
df_clean = df.copy()
bins = [17, 25, 35, 45, 60]

labels = [
    "Young Adult",
    "Adult",
    "Middle Age",
    "Senior Adult"
]

df_clean["Age_Group"] = pd.cut(
    df_clean["Customer_Age"],
    bins=bins,
    labels=labels
)

In [9]:
df_clean["Age_Group"].value_counts()

Age_Group
Middle Age      61715
Senior Adult    17386
Adult           11653
Young Adult      9246
Name: count, dtype: int64

In [10]:
df_clean.groupby(
    "Age_Group",
    observed=True
)["Order_Value"].mean().sort_values(
    ascending=False
)

Age_Group
Adult           1806.888441
Senior Adult    1786.419418
Middle Age      1784.018780
Young Adult     1782.305970
Name: Order_Value, dtype: float64

In [11]:
df_clean["Delivery_Performance"] = pd.cut(
    df_clean["Delivery_Time_Min"],
    bins=[0, 45, 120, 210, float("inf")],
    labels=[
        "Fast",
        "Moderate",
        "Slow",
        "Very Slow"
    ]
)

In [12]:
df_clean["Delivery_Performance"].value_counts()

Delivery_Performance
Moderate     49691
Fast         17063
Slow         16744
Very Slow    16502
Name: count, dtype: int64

In [14]:
df_clean["Order_Date"] = pd.to_datetime(
    df_clean["Order_Date"],
    errors="coerce"
)

In [15]:
df_clean.loc[
    df_clean["Order_Date"].notna(),
    "Order_Day"
] = np.where(
    df_clean.loc[
        df_clean["Order_Date"].notna(),
        "Order_Date"
    ].dt.dayofweek >= 5,
    "Weekend",
    "Weekday"
)

In [16]:
df_clean[
    [
        "Profit_Margin",
        "Profit_Margin_Percentage"
    ]
].head()

,Profit_Margin,Profit_Margin_Percentage
0,0.13,13.0
1,0.48,48.0
2,0.08,8.0
3,0.04,4.0
4,0.12,12.0


In [17]:
df_clean["Order_Month"] = (
    df_clean["Order_Date"]
    .dt.month_name()
)

In [18]:
df_clean["Order_Month_Number"] = (
    df_clean["Order_Date"].dt.month
)

In [19]:
monthly_revenue = (
    df_clean
    .dropna(subset=["Order_Date"])
    .groupby(
        ["Order_Month_Number", "Order_Month"]
    )["Final_Amount"]
    .sum()
    .reset_index()
    .sort_values("Order_Month_Number")
)

In [20]:
df_clean["Is_Cancelled"] = np.where(
    df_clean["Order_Status"] == "Cancelled",
    1,
    0
)

In [21]:
cancellation_rate = (
    df_clean["Is_Cancelled"].mean() * 100
)

print(cancellation_rate)

15.036


In [22]:
df_clean["Revenue"] = np.where(
    df_clean["Order_Status"] == "Delivered",
    df_clean["Final_Amount"],
    0
)

In [24]:
df_clean["Estimated_Profit"] = (
    df_clean["Revenue"]
    * df_clean["Profit_Margin"]
)

In [25]:
print("Shape:", df_clean.shape)

print(
    "\nDuplicates:",
    df_clean.duplicated().sum()
)

print(
    "\nRating > 5:",
    (df_clean["Restaurant_Rating"] > 5).sum()
)

print(
    "\nNegative Profit Margin:",
    (df_clean["Profit_Margin"] < 0).sum()
)

print(
    "\nNegative Final Amount:",
    (df_clean["Final_Amount"] < 0).sum()
)

print(
    "\nDiscount > Order Value:",
    (
        df_clean["Discount_Applied"]
        > df_clean["Order_Value"]
    ).sum()
)

print(
    "\nCancelled Orders With Rating:",
    df_clean.loc[
        df_clean["Order_Status"] == "Cancelled",
        "Delivery_Rating"
    ].notna().sum()
)

Shape: (100000, 33)

Duplicates: 0

Rating > 5: 0

Negative Profit Margin: 0

Negative Final Amount: 0

Discount > Order Value: 0

Cancelled Orders With Rating: 0


In [26]:
df_clean.isnull().sum()[
    df_clean.isnull().sum() > 0
]

Order_Date             1014
Delivery_Rating       15036
Order_Month            1014
Order_Month_Number     1014
dtype: int64

In [27]:
df_clean[
    [
        "Customer_Age",
        "Age_Group",
        "Order_Day",
        "Peak_Hour",
        "Delivery_Time_Min",
        "Delivery_Performance",
        "Profit_Margin_Percentage",
        "Order_Month",
        "Is_Cancelled",
        "Revenue",
        "Estimated_Profit"
    ]
].head(10)

,Customer_Age,Age_Group,Order_Day,Peak_Hour,Delivery_Time_Min,Delivery_Performance,Profit_Margin_Percentage,Order_Month,Is_Cancelled,Revenue,Estimated_Profit
0,19.0,Young Adult,Weekend,Peak,187.0,Slow,13.0,October,0,1147.0,149.11
1,39.0,Middle Age,Weekday,Peak,20.0,Fast,48.0,August,0,4849.0,2327.52
2,39.0,Middle Age,Weekend,Peak,207.0,Slow,8.0,December,0,737.0,58.96
3,39.0,Middle Age,Weekday,Unknown,143.0,Slow,4.0,October,1,0.0,0.00
4,57.0,Senior Adult,Weekend,Non-Peak,51.0,Moderate,12.0,February,0,352.0,42.24
5,37.0,Middle Age,Weekday,Peak,56.0,Moderate,24.0,July,0,897.0,215.28
6,40.0,Middle Age,Weekday,Unknown,273.0,Very Slow,2.0,February,0,1197.0,23.94
7,39.0,Middle Age,Weekend,Unknown,120.0,Moderate,18.0,May,0,3486.0,627.48
8,39.0,Middle Age,Weekend,Peak,120.0,Moderate,8.0,September,0,425.0,34.00
9,39.0,Middle Age,Weekday,Peak,120.0,Moderate,3.0,January,0,4641.0,139.23


In [ ]:
df_clean.to_csv(
    "online_food_delivery_cleaned_feature.csv",
    index=False
)